In [2]:
import sys
import os
os.environ['PROJ_DATA'] = "/pscratch/sd/p/plutzner/proj_data"
import xarray as xr
import torch
import torchinfo
import random
import numpy as np
import importlib as imp
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import cartopy.crs as ccrs
import json
import pickle
import gzip
import scipy
from scipy import stats
from cftime import DatetimeNoLeap
from datetime import datetime
from sklearn.metrics import mean_squared_error
#import matplotlib.colors as mcolorsxx

# %load_ext autoreload
# %autoreload 2
import utils
import utils.filemethods as filemethods
import databuilder.data_loader as data_loader
from utils.filemethods import open_data_file
from utils import utils

In [2]:
# open 0101, 0151, 0201: 
ens1_path = '/pscratch/sd/p/plutzner/E3SM/bigdata/input_vars.P_T_Z5.v2.LR.historical_0101.eam.h1.1850-2014.nc'
ens2_path = '/pscratch/sd/p/plutzner/E3SM/bigdata/input_vars.P_T_Z5.v2.LR.historical_0151.eam.h1.1850-2014.nc'
ens3_path = '/pscratch/sd/p/plutzner/E3SM/bigdata/input_vars.P_T_Z5.v2.LR.historical_0201.eam.h1.1850-2014.nc'

# ens1 = open_data_file(ens1_path)
ens2 = open_data_file(ens2_path)
# ens3 = open_data_file(ens3_path)

data = ens2

data['PRECT'] = data['PRECT'] * 86400000

# drop 'bnds' dimensions, drop 'time_bnds' 'lon_bnds' and 'lat_bnds' variables: 
data = data.drop_vars(['time_bnds', 'lon_bnds', 'lat_bnds'])
# data = data.drop_dims(['bnds'])

print(data)
print(data['PRECT'][50:65, ...])

# save file again as netcdf
data.to_netcdf('/pscratch/sd/p/plutzner/E3SM/bigdata/input_vars.P_T_Z5.v2.LR.historical_0151.eam.h1.1850-2014_precip_mmday.nc', mode='w')

<xarray.Dataset> Size: 62GB
Dimensions:  (time: 60226, lon: 360, lat: 180)
Coordinates:
  * time     (time) object 482kB 1850-01-01 00:00:00 ... 2015-01-01 00:00:00
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables:
    PRECT    (time, lat, lon) float64 31GB 0.8474 0.8408 0.8341 ... 1.127 1.129
    TS       (time, lat, lon) float32 16GB ...
    Z500     (time, lat, lon) float32 16GB ...
Attributes: (12/29)
    CDI:                        Climate Data Interface version 2.4.4 (https:/...
    Conventions:                CF-1.7
    source:                     E3SM Atmosphere Model
    institution:                LLNL (Lawrence Livermore National Laboratory,...
    ne:                         30
    fv_nphys:                   2
    ...                         ...
    remap_hostname:             login15
    remap_version:              5.1.4
    nco_openmp_thread_number:

### Convert Z500 in ERA5 to correct units and save: 

In [4]:
ERA5_allvars = open_data_file('/pscratch/sd/p/plutzner/E3SM/bigdata/ERA5/ERA5_1x1_input_vars_P_TS_Z500_1940-2023_daily.nc')

In [6]:
print(ERA5_allvars['z'].values)

[[[51881.34  51881.34  51881.34  ... 51881.34  51881.34  51881.34 ]
  [51746.902 51743.59  51740.34  ... 51756.965 51753.652 51750.34 ]
  [51577.277 51571.34  51565.465 ... 51594.965 51589.027 51583.152]
  ...
  [49953.277 49954.652 49956.215 ... 49949.027 49950.34  49951.777]
  [50013.965 50014.527 50015.152 ... 50012.277 50012.84  50013.34 ]
  [50127.465 50127.465 50127.465 ... 50127.465 50127.465 50127.465]]

 [[50704.332 50704.332 50704.332 ... 50704.332 50704.332 50704.332]
  [50677.207 50673.395 50669.645 ... 50688.707 50684.832 50681.02 ]
  [50624.27  50617.145 50609.957 ... 50645.332 50638.27  50631.207]
  ...
  [49971.832 49973.395 49974.957 ... 49967.27  49968.77  49970.332]
  [50017.645 50018.332 50018.957 ... 50015.77  50016.332 50017.02 ]
  [50083.645 50083.645 50083.645 ... 50083.645 50083.645 50083.645]]

 [[49726.348 49726.348 49726.348 ... 49726.348 49726.348 49726.348]
  [49672.285 49669.66  49667.098 ... 49679.91  49677.41  49674.848]
  [49657.41  49651.348 49645.223

In [7]:
# convert ERA5 Z500 to m:
ERA5_allvars['z'] = ERA5_allvars['z'] / 9.81

# save file again as netcdf
ERA5_allvars.to_netcdf('/pscratch/sd/p/plutzner/E3SM/bigdata/ERA5/ERA5_1x1_input_vars_P_TS_Z500_1940-2023_daily_m.nc', mode='w')